In [52]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F

BRONZE_INPUT_BASE_PATH=(
    "/home/hadoop/workspace/docker/glue/output/bronze/customers/"
)

spark_context=SparkContext.getOrCreate()
glue_context=GlueContext(spark_context)
spark=glue_context.spark_session

spark.sparkContext.setLogLevel("WARN")

df=spark.read.parquet(BRONZE_INPUT_BASE_PATH)

In [53]:
df.show(truncate=False)

+-----------+----------+---------+-------------------------------+-------------+-------------+------+----------+-------------+-------+-------------------+-------------------+-------------+----------+--------------------------+---------------------+
|customer_id|first_name|last_name|email                          |phone        |date_of_birth|gender|city      |state        |country|created_timestamp  |updated_timestamp  |source_system|is_deleted|ingestion_timestamp       |source_table         |
+-----------+----------+---------+-------------------------------+-------------+-------------+------+----------+-------------+-------+-------------------+-------------------+-------------+----------+--------------------------+---------------------+
|1001       |Ikshita   |Biswas   |ikshita.biswas.1@example.com   |02784980841  |2005-04-15   |O     |Mysuru    |Karnataka    |India  |2023-08-12 07:31:04|2025-03-08 08:29:10|SQLSERVER    |false     |2026-07-17 13:05:00.309544|master_data.customers|
|100

In [54]:
df.count()

2000

In [55]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-----------+----------+---------+-----+-----+-------------+------+----+-----+-------+-----------------+-----------------+-------------+----------+-------------------+------------+
|customer_id|first_name|last_name|email|phone|date_of_birth|gender|city|state|country|created_timestamp|updated_timestamp|source_system|is_deleted|ingestion_timestamp|source_table|
+-----------+----------+---------+-----+-----+-------------+------+----+-----+-------+-----------------+-----------------+-------------+----------+-------------------+------------+
|          0|         0|        0|    0|  103|            0|     0|   0|    0|      0|                0|                0|            0|         0|                  0|           0|
+-----------+----------+---------+-----+-----+-------------+------+----+-----+-------+-----------------+-----------------+-------------+----------+-------------------+------------+



In [56]:
df.select(F.col("gender")).distinct().show()

+------+
|gender|
+------+
|     M|
|     F|
|     O|
+------+



In [57]:
df.select(F.col("country")).distinct().show()

+-------+
|country|
+-------+
|  India|
+-------+



In [58]:
df.select(F.col("state")).distinct().show()

+-------------+
|        state|
+-------------+
|      Gujarat|
|        Delhi|
|    Karnataka|
|    Telangana|
|    Rajasthan|
|  Maharashtra|
|   Tamil Nadu|
|      Haryana|
|  West Bengal|
|Uttar Pradesh|
+-------------+



## Clean and Standardize data

In [59]:
df=(
    df
    .withColumn("first_name",F.trim(F.initcap(F.col("first_name"))))
    .withColumn("last_name",F.trim(F.initcap(F.col("last_name"))))
    .withColumn("email",F.lower(F.trim(F.col("email"))))
    .withColumn("phone",F.regexp_replace(F.trim(F.col("phone")),r"[^0-9]",""))
    .withColumn("gender", 
                F.when(F.lower(F.trim(F.col("gender")))=="m",F.lit("MALE"))
                .when(F.lower(F.trim(F.col("gender")))=="f",F.lit("FEMALE"))
                .when(F.lower(F.trim(F.col("gender")))=="o",F.lit("OTHER"))
                .otherwise(F.upper(F.trim(F.col("gender"))))
               )
    .withColumn("city",F.trim(F.initcap(F.col("city"))))
    .withColumn("state",F.trim(F.initcap(F.col("state"))))
    .withColumn("country",F.trim(F.initcap(F.col("country"))))
    .withColumn("source_system",F.trim(F.initcap(F.col("source_system"))))
)
df.show()

+-----------+----------+---------+--------------------+------------+-------------+------+----------+-------------+-------+-------------------+-------------------+-------------+----------+--------------------+--------------------+
|customer_id|first_name|last_name|               email|       phone|date_of_birth|gender|      city|        state|country|  created_timestamp|  updated_timestamp|source_system|is_deleted| ingestion_timestamp|        source_table|
+-----------+----------+---------+--------------------+------------+-------------+------+----------+-------------+-------+-------------------+-------------------+-------------+----------+--------------------+--------------------+
|       1001|   Ikshita|   Biswas|ikshita.biswas.1@...| 02784980841|   2005-04-15| OTHER|    Mysuru|    Karnataka|  India|2023-08-12 07:31:04|2025-03-08 08:29:10|    Sqlserver|     false|2026-07-17 13:05:...|master_data.custo...|
|       1002|     Krish|    Lalla|krish.lalla.2@exa...|918244935348|   1978-10-1

## Find and drop invalid data

In [60]:
before_count=df.count()
print(f"Before count: {before_count}")
df=(
    df.withColumn("dq_errors",
                  F.array_compact(
                 F.array(
                    F.when(F.col("customer_id").isNull(), F.lit("CUSTOMER_ID_MISSING")),
                    F.when((F.col("first_name").isNull()) | (F.col("first_name") == ""), F.lit("FNAME_MISSING")),
                    F.when((F.col("last_name").isNull()) | (F.col("last_name") == ""), F.lit("LNAME_MISSING")),
                    F.when((F.col("email").isNull()) | (F.col("email") == ""), F.lit("email_MISSING")),
                    F.when((F.col("phone").isNull()) | (F.col("phone") == ""), F.lit("phone_MISSING")),
                    F.when((F.col("date_of_birth").isNull()) | (F.col("date_of_birth") == ""), F.lit("date_of_birth_MISSING")),
                    F.when((F.col("gender").isNull()) | (F.col("gender") == ""), F.lit("gender_MISSING")),
                    F.when((F.col("city").isNull()) | (F.col("city") == ""), F.lit("city_MISSING")),
                    F.when((F.col("state").isNull()) | (F.col("state") == ""), F.lit("state_MISSING")),
                    F.when((F.col("country").isNull()) | (F.col("country") == ""), F.lit("country_MISSING")),
                    F.when((F.col("created_timestamp").isNull()) | (F.col("created_timestamp") == ""), F.lit("created_timestamp_MISSING")),
                    F.when((F.col("updated_timestamp").isNull()) | (F.col("updated_timestamp") == ""), F.lit("updated_timestamp_MISSING")),
                    F.when((F.col("source_system").isNull()) | (F.col("source_system") == ""), F.lit("source_system_MISSING")),
                    F.when((F.col("is_deleted").isNull()) | (F.col("is_deleted") == ""), F.lit("is_deleted_MISSING")),
                    F.when((F.col("ingestion_timestamp").isNull()) | (F.col("ingestion_timestamp") == ""), F.lit("ingestion_timestamp_MISSING")),
                    F.when((F.col("source_table").isNull()) | (F.col("source_table") == ""), F.lit("source_table_MISSING"))
                 )))
)
cleaned_df=df.filter(F.size(F.col("dq_errors"))==0).drop(F.col("dq_errors"))
rejectd_df=df.filter(F.size(F.col("dq_errors"))>0)

cleaned_df_count=cleaned_df.count()
rejected_df_count=rejectd_df.count()
print(f"cleaned_df_count: {cleaned_df_count}")
print(f"rejected_df_count: {rejected_df_count}")

Before count: 2000
cleaned_df_count: 1897
rejected_df_count: 103


In [61]:
rejectd_df.show()

+-----------+----------+-----------+--------------------+-----+-------------+------+----------+-----------+-------+-------------------+-------------------+-------------+----------+--------------------+--------------------+---------------+
|customer_id|first_name|  last_name|               email|phone|date_of_birth|gender|      city|      state|country|  created_timestamp|  updated_timestamp|source_system|is_deleted| ingestion_timestamp|        source_table|      dq_errors|
+-----------+----------+-----------+--------------------+-----+-------------+------+----------+-----------+-------+-------------------+-------------------+-------------+----------+--------------------+--------------------+---------------+
|       1032|  Vyanjana|      Borah|vyanjana.borah.32...| NULL|   1986-10-17| OTHER|    Jaipur|  Rajasthan|  India|2026-06-11 02:10:02|2026-07-11 15:31:28|    Sqlserver|     false|2026-07-17 13:05:...|master_data.custo...|[phone_MISSING]|
|       1097|     Pooja|    Ganesan|pooja.ga